# Google Colab setup
Before running this notebook, choose **Runtime → Change runtime type → T4 GPU**. This notebook mounts Drive, clones or updates the repository, installs the package without replacing Colab's GPU-enabled PyTorch, links persistent experiment storage, verifies the dataset, and creates/reuses the seed-42 split.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/jtackerman1/Applied_AI_Midterm_SRGAN.git"
REPOSITORY_DIR = Path("/content/Applied_AI_Midterm_SRGAN")
if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML>=6.0.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR), "--no-deps"], check=True)
print(f"Repository ready at {REPOSITORY_DIR}")

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select Runtime → Change runtime type → T4 GPU, then rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Persistent Drive layout
Upload the local `data/raw` directory to `MyDrive/Applied_AI_Midterm_SRGAN_runtime/data/raw`. At minimum, Colab must see `data/raw/train/cats` and `data/raw/train/dogs`. Vendor `validation` and `test` folders may also be present, but the project intentionally ignores them and creates its own 70/30 split.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Applied_AI_Midterm_SRGAN_runtime")
persistent_links = {
    REPOSITORY_DIR / "data/raw": DRIVE_ROOT / "data/raw",
    REPOSITORY_DIR / "data/splits": DRIVE_ROOT / "data/splits",
    REPOSITORY_DIR / "data/generated": DRIVE_ROOT / "data/generated",
    REPOSITORY_DIR / "checkpoints": DRIVE_ROOT / "checkpoints",
    REPOSITORY_DIR / "artifacts": DRIVE_ROOT / "artifacts",
}
for link_path, drive_path in persistent_links.items():
    drive_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(drive_path, target_is_directory=True)
    print(f"{link_path.relative_to(REPOSITORY_DIR)} -> {drive_path}")

In [ ]:
from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import discover_images, prepare_splits

config = load_config(REPOSITORY_DIR / "configs/config.yaml")
try:
    discovered = discover_images(REPOSITORY_DIR / "data/raw")
except (FileNotFoundError, ValueError) as error:
    raise RuntimeError(
        "Dataset is not ready in Drive. Upload it to "
        f"{DRIVE_ROOT / 'data/raw'} and rerun this cell. Original error: {error}"
    ) from error
display(discovered.groupby(["class_name", "label"]).size().rename("source_images").to_frame())
splits = prepare_splits(
    REPOSITORY_DIR / "data/raw",
    REPOSITORY_DIR / "data/splits",
    train_ratio=config.train_ratio,
    random_seed=config.random_seed,
)
display({"training_examples": len(splits.train), "reserved_test_examples": len(splits.test)})

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPOSITORY_DIR, check=True)
print("Setup complete. Keep this runtime connected and run notebooks 01 through 06 in order.")

## Execution and resume order
1. `01_data_preparation.ipynb` — inspect the already persisted split.
2. `02_model_a_transfer_learning.ipynb` — train Model A.
3. `03_srgan_training.ipynb` — pretrain and adversarially train SRGAN. This is the long stage; rerun setup and notebook 03 after a disconnect to resume from Drive checkpoints.
4. `04_generate_srgan_training_images.ipynb` — generate training images and manifest.
5. `05_model_b_transfer_learning.ipynb` — train Model B.
6. `06_final_comparison.ipynb` — produce measured results.

Do not delete `MyDrive/Applied_AI_Midterm_SRGAN_runtime`; it contains all checkpoints and outputs.